# Build `nmdc_metadata.biosample_to_gene`

Precomputes the (biosample, annotation workflow run) mapping with provenance
flags for each MaterialProcessing type in the chain.

See `docs/biosample_to_gene_table.md` for schema, example queries, and
maintenance instructions — especially what to do when new MaterialProcessing
subclasses are added to the NMDC schema.

**Run this notebook after each NMDC data load.**

### Session setup

In [ ]:
import pandas as pd
import time

conn = get_trino_connection()
print("Trino connection ready")

### Configuration

**MAINTENANCE NOTE:** `PROCESSING_TYPES` is hardcoded against the MaterialProcessing
subclasses present in BERDL as of 2026-04-30. If a new subclass is added to the
NMDC schema and loaded into BERDL, add it here before rebuilding.

To check for new types: run the preflight cell below and compare its output to
the keys of `PROCESSING_TYPES`.

In [ ]:
PROCESSING_TYPES = {
    'nmdc:Extraction':                        'has_extraction',
    'nmdc:LibraryPreparation':                'has_library_prep',
    'nmdc:SubSamplingProcess':                'has_subsampling',
    'nmdc:Pooling':                           'has_pooling',
    'nmdc:ChromatographicSeparationProcess':  'has_chromatographic_separation',
    'nmdc:DissolvingProcess':                 'has_dissolving',
    'nmdc:ChemicalConversionProcess':         'has_chemical_conversion',
    'nmdc:FiltrationProcess':                 'has_filtration',
}

ANNOTATION_WORKFLOW_TYPES = (
    'nmdc:MetagenomeAnnotation',
    'nmdc:MetaproteomicsAnalysis',
    'nmdc:MetabolomicsAnalysis',
    'nmdc:NOMAnalysis',
    'nmdc:MetagenomeAssembly',
    'nmdc:MetatranscriptomeAnnotation',
)

BATCH_SIZE = 500   # workflow runs per Trino query; lower if TOO_MANY_REQUESTS_FAILED
MAX_DEPTH  = 15

TARGET_SCHEMA = 'nmdc_metadata'
TARGET_TABLE  = 'biosample_to_gene'

### Preflight: check for unknown MaterialProcessing types

If any type appears here that is not in `PROCESSING_TYPES`, add it to the
config cell above before proceeding.

In [ ]:
cur = conn.cursor()
cur.execute("""
    SELECT type, COUNT(*) AS n
    FROM   nmdc_metadata.material_processing_set
    GROUP  BY type
    ORDER  BY n DESC
""")
observed = pd.DataFrame(cur.fetchall(), columns=["type", "n"])

unknown = set(observed["type"]) - set(PROCESSING_TYPES.keys())
if unknown:
    print(f"WARNING: unknown MaterialProcessing types not in PROCESSING_TYPES:")
    for t in sorted(unknown):
        print(f"  {t}")
    print("Add them to PROCESSING_TYPES before continuing.")
else:
    print("OK: all MaterialProcessing types accounted for")
observed

### Edge subquery and helpers

In [ ]:
_EDGES = """
    SELECT parent_id  AS src, was_informed_by AS next_id
    FROM   nmdc_metadata.workflow_execution_set_was_informed_by
    UNION ALL
    SELECT parent_id  AS src, has_input       AS next_id
    FROM   nmdc_metadata.data_generation_set_has_input
    UNION ALL
    SELECT has_output AS src, parent_id       AS next_id
    FROM   nmdc_metadata.material_processing_set_has_output
    UNION ALL
    SELECT parent_id  AS src, has_input       AS next_id
    FROM   nmdc_metadata.material_processing_set_has_input
"""

def _walk_batch(cur, workflow_ids, max_depth=MAX_DEPTH):
    """Single recursive CTE that returns both biosample endpoints and
    MaterialProcessing types visited.  Two result sets in one query pass."""
    values = ",\n        ".join(f"('{wid}')" for wid in workflow_ids)
    cur.execute(f"""
        WITH RECURSIVE upstream(origin, id, depth) AS (
            SELECT CAST(id AS VARCHAR), CAST(id AS VARCHAR), CAST(0 AS BIGINT)
            FROM (VALUES
                {values}
            ) AS t(id)
            UNION ALL
            SELECT u.origin, e.next_id, u.depth + 1
            FROM   upstream u
            JOIN ({_EDGES}) e ON e.src = u.id
            WHERE  u.depth < {max_depth}
              AND  u.id NOT LIKE 'nmdc:bsm%'
        )
        -- Biosample endpoints
        SELECT 'bsm'         AS row_type,
               origin        AS workflow_run_id,
               id            AS value,
               depth         AS n_hops
        FROM   upstream
        WHERE  id LIKE 'nmdc:bsm%'

        UNION ALL

        -- MaterialProcessing types for all intermediate nodes
        SELECT 'mp'          AS row_type,
               u.origin      AS workflow_run_id,
               mp.type       AS value,
               u.depth       AS n_hops
        FROM   upstream u
        JOIN   nmdc_metadata.material_processing_set mp ON mp.id = u.id
    """)
    rows = cur.fetchall()
    df = pd.DataFrame(rows, columns=["row_type", "workflow_run_id", "value", "n_hops"])
    bsm = (df[df["row_type"] == "bsm"][["workflow_run_id", "value", "n_hops"]]
           .rename(columns={"value": "biosample_id"})
           .copy())
    mp  = (df[df["row_type"] == "mp"][["workflow_run_id", "value"]]
           .rename(columns={"value": "processing_type"})
           .drop_duplicates()
           .copy())
    return bsm, mp

### Step 1: collect annotation workflow runs

In [ ]:
t0 = time.monotonic()
cur = conn.cursor()
types_sql = ", ".join(f"'{t}'" for t in ANNOTATION_WORKFLOW_TYPES)
cur.execute(f"""
    SELECT id, type
    FROM   nmdc_metadata.workflow_execution_set
    WHERE  type IN ({types_sql})
""")
wf_df = pd.DataFrame(cur.fetchall(), columns=["workflow_run_id", "workflow_type"])
print(f"{len(wf_df)} annotation workflow runs  ({time.monotonic()-t0:.1f}s)")
wf_df["workflow_type"].value_counts()

### Step 2: recursive walk in batches

In [ ]:
all_bsm = []
all_mp  = []
ids = wf_df["workflow_run_id"].tolist()
n_batches = (len(ids) + BATCH_SIZE - 1) // BATCH_SIZE

for i in range(0, len(ids), BATCH_SIZE):
    batch = ids[i:i + BATCH_SIZE]
    t1 = time.monotonic()
    bsm, mp = _walk_batch(cur, batch)
    all_bsm.append(bsm)
    all_mp.append(mp)
    print(f"Batch {i//BATCH_SIZE+1}/{n_batches}: "
          f"{len(bsm)} bsm pairs, {len(mp)} mp hits  "
          f"({time.monotonic()-t1:.1f}s)")

r2b = (pd.concat(all_bsm, ignore_index=True)
         .groupby(["workflow_run_id", "biosample_id"], as_index=False)["n_hops"]
         .min())
mp_all = pd.concat(all_mp, ignore_index=True).drop_duplicates()
print(f"\nTotal: {len(r2b)} (workflow_run_id, biosample_id) pairs  "
      f"({time.monotonic()-t0:.1f}s)")

### Step 3: pivot processing types to boolean columns

In [ ]:
# Pivot: one row per workflow_run_id, one column per processing type
mp_pivot = (
    mp_all.assign(flag=True)
          .pivot_table(index="workflow_run_id", columns="processing_type",
                       values="flag", aggfunc="any", fill_value=False)
          .reset_index()
)
mp_pivot.columns.name = None

# Rename to snake_case flag names; add missing columns as False
mp_pivot = mp_pivot.rename(columns=PROCESSING_TYPES)
for col in PROCESSING_TYPES.values():
    if col not in mp_pivot.columns:
        mp_pivot[col] = False

# Merge everything together
result = (
    r2b
    .merge(wf_df, on="workflow_run_id", how="left")
    .merge(mp_pivot[["workflow_run_id"] + list(PROCESSING_TYPES.values())],
           on="workflow_run_id", how="left")
)
for col in PROCESSING_TYPES.values():
    result[col] = result[col].fillna(False)

print(f"{len(result)} rows, {len(result.columns)} columns")
result.head()

### Step 4: write directly to Silver as a Delta table

In [ ]:
spark = get_spark_session(app_name="build_biosample_to_gene", tenant_name="nmdc")

(spark.createDataFrame(result)
      .write
      .format("delta")
      .mode("overwrite")
      .option("overwriteSchema", "true")
      .saveAsTable(f"{TARGET_SCHEMA}.{TARGET_TABLE}"))

print(f"Wrote {TARGET_SCHEMA}.{TARGET_TABLE}")
print(f"Total build time: {time.monotonic()-t0:.1f}s")

### Step 5: verify

In [ ]:
spark.sql(f"""
    SELECT workflow_type,
           COUNT(DISTINCT biosample_id) AS n_biosamples,
           COUNT(DISTINCT workflow_run_id) AS n_workflow_runs,
           ROUND(AVG(n_hops), 1) AS avg_hops,
           SUM(CAST(has_pooling AS INT)) AS pooled_pairs,
           SUM(CAST(has_extraction AS INT)) AS extracted_pairs
    FROM   {TARGET_SCHEMA}.{TARGET_TABLE}
    GROUP  BY workflow_type
    ORDER  BY n_workflow_runs DESC
""").toPandas()